# GAN Imputation v2 — Finetuned CNN Ensemble (TensorFlow)
## Final Architecture & Settings
- **Framework:** TensorFlow 2.x (`import tensorflow as tf`)
- **Generator & Discriminator:** Finetuned CNN (Conv1D 32→64→64→32)
- **Regularisation:** Dropout=0.25, L2=1e-4, Alpha=5.0
- **Training:** Early stopping on validation reconstruction (patience=60)
- **Ensemble:** 5 models averaged (seeds 42, 123, 456, 789, 1024)
- **Metrics:** NRMSE%, NMAE%, R²%

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {tf.keras.__version__}")

STATIONS   = ["nada", "lembing", "reman"]
N_STATIONS = 3
WINDOW     = 15
HALF_W     = 7
BATCH_SIZE = 128
N_EPOCHS   = 600
PATIENCE   = 60
ALPHA      = 5.0
HINT_RATE  = 0.9
LR         = 5e-4
L2_REG     = 1e-4
DROPOUT    = 0.25
SEEDS      = [42, 123, 456, 789, 1024]

---
## 1. Load Daily Rainfall Data

In [ ]:
PATH = "../data/raw/BulkExport-0570061RF,0570051RF,0570021RF-20251006110824.csv"
df_raw = pd.read_csv(PATH, skiprows=4, parse_dates=[0, 1])
df_raw.rename(columns={
    "Start of Interval (UTC+08:00)": "date",
    "End of Interval (UTC+08:00)":   "date_end",
    "Total (mm)":                    "nada",
    "Total (mm).1":                  "lembing",
    "Total (mm).2":                  "reman",
}, inplace=True)

df = df_raw[["date"] + STATIONS].copy().sort_values("date").reset_index(drop=True)
print(f"Rows: {len(df):,} | Range: {df['date'].min().date()} to {df['date'].max().date()}")
print("\nMissing values:")
for s in STATIONS:
    n = df[s].isna().sum()
    print(f"  {s:<10}: {n:>4}  ({100*n/len(df):.2f}%)")
total = df[STATIONS].isna().sum().sum()
print(f"  {'Overall':<10}: {total:>4}  ({100*total/(len(df)*N_STATIONS):.2f}%)")
df.head()

---
## 2. Preprocessing
### 2.1 Log1p + MinMax Normalisation

In [ ]:
data_np   = df[STATIONS].values.copy()
mask      = (~np.isnan(data_np)).astype(np.float32)
data_log  = np.where(np.isnan(data_np), 0.0, np.log1p(data_np)).astype(np.float32)

scaler    = MinMaxScaler((0, 1))
scaler.fit(data_log[mask.astype(bool)].reshape(-1, 1))

data_norm = np.zeros_like(data_log, dtype=np.float32)
for j in range(N_STATIONS):
    idx = mask[:, j] == 1
    data_norm[idx, j] = scaler.transform(data_log[idx, j].reshape(-1, 1)).ravel()

print(f"Observed : {int(mask.sum()):,}  ({100*mask.mean():.1f}%)")
print(f"Missing  : {int((1-mask).sum()):,}  ({100*(1-mask).mean():.1f}%)")
print(f"Log range: [{data_log[mask==1].min():.3f}, {data_log[mask==1].max():.3f}]")

### 2.2 Validation Holdout (20%)

In [ ]:
rng      = np.random.RandomState(42)
val_mask = np.zeros_like(mask)
for j in range(N_STATIONS):
    obs = np.where(mask[:, j] == 1)[0]
    val_mask[rng.choice(obs, int(len(obs)*0.2), replace=False), j] = 1

val_true   = data_norm.copy()
train_mask = mask - val_mask
train_data = data_norm * train_mask

print(f"Original observed : {int(mask.sum()):,}")
print(f"Held out for val  : {int(val_mask.sum()):,}")
print(f"Training observed : {int(train_mask.sum()):,}")

---
## 3. Temporal Window Dataset

In [ ]:
def build_windows(data, m):
    N  = len(data)
    Xw = np.zeros((N, WINDOW, N_STATIONS), np.float32)
    Mw = np.zeros((N, WINDOW, N_STATIONS), np.float32)
    for i in range(N):
        s=max(0,i-HALF_W); e=min(N,i+HALF_W+1); o=HALF_W-(i-s); l=e-s
        Xw[i,o:o+l]=data[s:e]; Mw[i,o:o+l]=m[s:e]
    return Xw, Mw

Xtr, Mtr = build_windows(train_data, train_mask)
Xfu, Mfu = build_windows(data_norm,  train_mask)
Xfu_tf   = tf.constant(Xfu)
Mfu_tf   = tf.constant(Mfu)
print(f"Window shape: {Xtr.shape}  (samples, timesteps, stations)")

---
## 4. Finetuned CNN Architecture

**Why these settings beat the original CNN:**

| Setting | Original CNN | Finetuned CNN | Reason |
|---|---|---|---|
| Filters | 64→128→128→64 | **32→64→64→32** | Smaller = less overfitting |
| Dropout | 0.1 | **0.25** | More regularisation |
| L2 weight decay | None | **1e-4** | Penalises large weights |
| Alpha | 10.0 | **5.0** | Less pressure to memorise observed |
| Training | 800 fixed epochs | **Early stop (patience=60)** | Stops before overfitting |

In [ ]:
def build_gen(seed=42):
    tf.random.set_seed(seed); np.random.seed(seed)
    reg = tf.keras.regularizers.l2(L2_REG)
    d   = tf.keras.Input((WINDOW, N_STATIONS), name="data")
    m   = tf.keras.Input((WINDOW, N_STATIONS), name="mask")
    x   = tf.keras.layers.Concatenate(axis=-1)([d * m, m])
    for f in [32, 64, 64, 32]:
        x = tf.keras.layers.Conv1D(f, 3, padding="same", kernel_regularizer=reg)(x)
        x = tf.keras.layers.LeakyReLU(0.2)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(DROPOUT)(x)
    out = tf.keras.layers.Conv1D(N_STATIONS, 1, padding="same", activation="sigmoid")(x)
    return tf.keras.Model([d, m], out, name=f"Generator_seed{seed}")

def build_disc(seed=42):
    tf.random.set_seed(seed); np.random.seed(seed)
    reg  = tf.keras.regularizers.l2(L2_REG)
    xi   = tf.keras.Input((WINDOW, N_STATIONS), name="x_hat")
    hi   = tf.keras.Input((WINDOW, N_STATIONS), name="hint")
    x    = tf.keras.layers.Concatenate(axis=-1)([xi, hi])
    for f in [32, 64, 64, 32]:
        x = tf.keras.layers.Conv1D(f, 3, padding="same", kernel_regularizer=reg)(x)
        x = tf.keras.layers.LeakyReLU(0.2)(x)
        x = tf.keras.layers.Dropout(DROPOUT)(x)
    out  = tf.keras.layers.Conv1D(N_STATIONS, 1, padding="same", activation="sigmoid")(x)
    return tf.keras.Model([xi, hi], out, name=f"Discriminator_seed{seed}")

# Show architecture for one model
gen_sample = build_gen(42)
gen_sample.summary()
print(f"\nParameters: {gen_sample.count_params():,} (vs 388K in original CNN)")

---
## 5. Training Utilities

In [ ]:
def bce(y_true, y_pred):
    """Element-wise BCE — output shape matches input (batch, 15, 3)."""
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    return -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))

def cosine_lr(ep, T=N_EPOCHS, base=LR, eta=1e-5):
    return eta + 0.5 * (base - eta) * (1 + np.cos(np.pi * ep / T))

def inv_transform(x):
    return np.maximum(np.expm1(scaler.inverse_transform(x.reshape(-1,1)).reshape(x.shape)), 0.0)

def get_center_preds(gen, Xw_tf, Mw_tf):
    out = []
    for s in range(0, Xw_tf.shape[0], BATCH_SIZE):
        o = gen([Xw_tf[s:s+BATCH_SIZE], Mw_tf[s:s+BATCH_SIZE]], training=False)
        out.append(o.numpy())
    return np.concatenate(out)[:, HALF_W, :]

def val_recon_score(gen):
    preds = get_center_preds(gen, Xfu_tf, Mfu_tf)
    err = 0.0; n = 0
    for j in range(N_STATIONS):
        vi = val_mask[:, j] == 1
        err += np.sum((preds[vi, j] - val_true[vi, j])**2); n += vi.sum()
    return err / (n + 1e-8)

def train_one_model(seed, Xw, Mw):
    tf.random.set_seed(seed); np.random.seed(seed)
    gen  = build_gen(seed); disc = build_disc(seed)
    opt_G = tf.keras.optimizers.Adam(LR, beta_1=0.5)
    opt_D = tf.keras.optimizers.Adam(LR, beta_1=0.5)

    @tf.function
    def step(bd, bm):
        hint = bm * tf.cast(tf.random.uniform(tf.shape(bm)) < HINT_RATE, tf.float32)
        with tf.GradientTape() as td:
            Go = gen([bd,bm],training=False); Xh = bd*bm + Go*(1-bm)
            Do = disc([Xh,hint],training=True)
            dl = tf.reduce_mean(bce(bm, Do) * (hint + (1-hint)*0.5))
        opt_D.apply_gradients(zip(td.gradient(dl, disc.trainable_variables), disc.trainable_variables))
        with tf.GradientTape() as tg:
            Go = gen([bd,bm],training=True); Xh = bd*bm + Go*(1-bm)
            Do = disc([Xh,hint],training=False)
            ga = tf.reduce_mean(-bce(bm, Do) * (1 - bm))
            rc = tf.reduce_sum((Go-bd)**2 * bm) / (tf.reduce_sum(bm) + 1e-8)
        opt_G.apply_gradients(zip(tg.gradient(ga+ALPHA*rc, gen.trainable_variables), gen.trainable_variables))

    idx = np.arange(len(Xw))
    bvr = float("inf"); bw = None; pat = 0
    for ep in range(N_EPOCHS):
        opt_G.learning_rate.assign(cosine_lr(ep))
        opt_D.learning_rate.assign(cosine_lr(ep))
        np.random.shuffle(idx)
        for s in range(0, len(idx), BATCH_SIZE):
            bi = idx[s:s+BATCH_SIZE]
            step(tf.constant(Xw[bi]), tf.constant(Mw[bi]))
        vr = val_recon_score(gen)
        if vr < bvr: bvr = vr; bw = gen.get_weights(); pat = 0
        else: pat += 1
        if pat >= PATIENCE:
            print(f"    Early stop ep {ep+1}  (val_recon={bvr:.6f})")
            break
    gen.set_weights(bw)
    return gen, bvr

print("Utilities ready.")

---
## 6. Ensemble Training (5 Models)

Five independent models are trained with different random seeds. Their outputs are averaged at inference time, reducing variance in the imputed values — especially helpful for ambiguous missing positions.

In [ ]:
print(f"Training {len(SEEDS)} models with seeds: {SEEDS}")
print("="*60)

val_preds = []
val_recons = []
history_all = []

for i, seed in enumerate(SEEDS):
    print(f"\nModel {i+1}/{len(SEEDS)}  (seed={seed})")
    gen, bvr = train_one_model(seed, Xtr, Mtr)
    preds = get_center_preds(gen, Xfu_tf, Mfu_tf)
    val_preds.append(preds)
    val_recons.append(bvr)
    print(f"  Best val recon: {bvr:.6f}")

print(f"\nMean val recon across ensemble: {np.mean(val_recons):.6f}")

---
## 7. Validation Results — Percentage Metrics

In [ ]:
def evaluate(imp_norm, label):
    imp_orig  = inv_transform(imp_norm)
    true_orig = inv_transform(val_true)
    print(f"\n{label}")
    print(f"  {'Station':<12} {'RMSE mm':<10} {'NRMSE%':<10} {'MAE mm':<10} {'NMAE%':<10} {'R²%'}")
    print("  " + "-"*60)
    all_t, all_p = [], []
    res = {}
    for j, s in enumerate(STATIONS):
        vi = val_mask[:, j] == 1
        yt = true_orig[vi, j]; yp = imp_orig[vi, j]
        rmse = np.sqrt(mean_squared_error(yt, yp))
        mae  = mean_absolute_error(yt, yp)
        r2   = r2_score(yt, yp)
        mo   = yt.mean()
        res[s] = (rmse, mae, r2)
        print(f"  {s:<12} {rmse:<10.2f} {rmse/mo*100:<10.1f} {mae:<10.2f} {mae/mo*100:<10.1f} {r2*100:.2f}")
        all_t.append(yt); all_p.append(yp)
    at = np.concatenate(all_t); ap = np.concatenate(all_p)
    rmse = np.sqrt(mean_squared_error(at,ap)); mae=mean_absolute_error(at,ap); r2=r2_score(at,ap); mo=at.mean()
    print("  " + "-"*60)
    print(f"  {'OVERALL':<12} {rmse:<10.2f} {rmse/mo*100:<10.1f} {mae:<10.2f} {mae/mo*100:<10.1f} {r2*100:.2f}")
    return rmse, mae, r2, res

# Single model baseline (seed=42)
single_norm = data_norm.copy()
for i2 in range(len(data_norm)):
    for j in range(N_STATIONS):
        if train_mask[i2, j] == 0:
            single_norm[i2, j] = np.clip(val_preds[0][i2, j], 0, 1)

# Ensemble average
ens_avg  = np.mean(val_preds, axis=0)
ens_norm = data_norm.copy()
for i2 in range(len(data_norm)):
    for j in range(N_STATIONS):
        if train_mask[i2, j] == 0:
            ens_norm[i2, j] = np.clip(ens_avg[i2, j], 0, 1)

print("=" * 65)
print("VALIDATION RESULTS  (20% holdout)")
print("=" * 65)
s_rmse, s_mae, s_r2, _ = evaluate(single_norm, "Single model (seed=42)")
e_rmse, e_mae, e_r2, e_res = evaluate(ens_norm, "Ensemble x5 (averaged)")

print("\n" + "="*65)
print("FINAL COMPARISON")
print("="*65)
print(f"{'Model':<30} {'RMSE':>8} {'MAE':>8} {'R²%':>8}")
print(f"{'MLP v1 (PyTorch)':<30} {'18.00':>8} {'7.62':>8} {'-5.56':>8}")
print(f"{'CNN single finetuned':<30} {s_rmse:>8.2f} {s_mae:>8.2f} {s_r2*100:>8.2f}")
print(f"{'CNN ensemble x5':<30} {e_rmse:>8.2f} {e_mae:>8.2f} {e_r2*100:>8.2f}")

### 7.1 Scatter — Actual vs Imputed (Ensemble)

In [ ]:
imp_orig  = inv_transform(ens_norm)
true_orig = inv_transform(val_true)
slabels   = ["Ldg. Nada", "Sg. Lembing", "Ldg. Kuala Reman"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, j, s, label in zip(axes, range(N_STATIONS), STATIONS, slabels):
    vi  = val_mask[:, j] == 1
    yt  = true_orig[vi, j]; yp = imp_orig[vi, j]
    ax.scatter(yt, yp, alpha=0.3, s=10, color="steelblue")
    mv  = max(yt.max(), yp.max()) * 1.1
    ax.plot([0, mv], [0, mv], "r--", lw=1, label="Perfect")
    r2  = e_res[s][2]; rmse = e_res[s][0]; mo = yt.mean()
    ax.set_title(f"{label}\nNRMSE={rmse/mo*100:.1f}%   R²={r2*100:.1f}%", fontweight="bold")
    ax.set_xlabel("Actual (mm)"); ax.set_ylabel("Imputed (mm)")
    ax.legend(); ax.set_xlim(0, mv); ax.set_ylim(0, mv)
plt.suptitle("CNN Ensemble — Actual vs Imputed (Validation Set)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../figures/gan/gain_cnn_ensemble_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, j, s, label in zip(axes, range(N_STATIONS), STATIONS, slabels):
    obs_idx = mask[:, j] == 1; mis_idx = mask[:, j] == 0
    obs_v   = inv_transform(data_norm)[obs_idx, j]
    imp_v   = imp_orig[mis_idx, j]
    obs_nz  = obs_v[obs_v > 0.1]; imp_nz = imp_v[imp_v > 0.1]
    if len(obs_nz): ax.hist(obs_nz, bins=50, alpha=0.5, density=True, color="steelblue", label="Observed")
    if len(imp_nz): ax.hist(imp_nz, bins=50, alpha=0.5, density=True, color="coral",     label="Imputed")
    ax.set_xlabel("Rainfall (mm)"); ax.set_ylabel("Density")
    ax.set_title(f"{label} (non-zero only)", fontweight="bold"); ax.legend()
plt.suptitle("Distribution: Observed vs CNN Ensemble Imputed", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../figures/gan/gain_cnn_ensemble_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Final Imputation — Retrain on All Observed Data
All 5 models are retrained using the full mask (no holdout) to maximise information for the actual imputation.

In [ ]:
print(f"Retraining {len(SEEDS)} models on full observed data...")
Xfu2, Mfu2 = build_windows(data_norm, mask)
Xfu2_tf    = tf.constant(Xfu2)
Mfu2_tf    = tf.constant(Mfu2)

final_preds = []
for i, seed in enumerate(SEEDS):
    print(f"  Model {i+1}/{len(SEEDS)}  (seed={seed})", end="")
    gen_f, bvr_f = train_one_model(seed, Xfu2, Mfu2)
    out = get_center_preds(gen_f, Xfu2_tf, Mfu2_tf)
    final_preds.append(out)
    print(f"  val_recon={bvr_f:.6f}")

final_avg  = np.mean(final_preds, axis=0)
final_norm = data_norm.copy()
for i2 in range(len(data_norm)):
    for j in range(N_STATIONS):
        if mask[i2, j] == 0:
            final_norm[i2, j] = np.clip(final_avg[i2, j], 0, 1)

final_imp = inv_transform(final_norm)
final_imp = np.round(final_imp, 1)
for j in range(N_STATIONS):
    obs_idx = mask[:, j] == 1
    final_imp[obs_idx, j] = df[STATIONS[j]].values[obs_idx]

print(f"\nNaN remaining: {np.isnan(final_imp).sum()}")
print(f"\n{'Station':<10} {'Orig mean':>10} {'New mean':>10} {'Shift%':>8}")
for j, s in enumerate(STATIONS):
    orig = df[s].dropna().mean(); new = np.nanmean(final_imp[:, j])
    print(f"{s:<10} {orig:>10.2f} {new:>10.2f} {abs(new-orig)/orig*100:>7.1f}%")

---
## 9. Export Artifacts

In [ ]:
df_complete = df.copy()
for j, s in enumerate(STATIONS):
    df_complete[s] = final_imp[:, j]
    df_complete[f"{s}_imputed"] = (mask[:, j] == 0).astype(int)

CSV_PATH = "../data/processed/completed_daily_rainfall_cnn_tf.csv"
df_complete.to_csv(CSV_PATH, index=False)

print(f"Completed dataset : {CSV_PATH}")
print(f"  Rows : {df_complete.shape[0]:,}  |  Cols : {df_complete.shape[1]}")
print(f"  Size : {os.path.getsize(CSV_PATH)/1024:.1f} KB")
print(f"  NaN  : {df_complete[STATIONS].isna().sum().sum()}")
df_complete.head()

---
## Summary

### Model Performance

| Model | NRMSE% | NMAE% | R²% |
|---|---|---|---|
| MLP v1 — PyTorch (original) | 18.00 | 7.62 | -5.56% |
| CNN overfit — TF (800 epochs) | 44.13 | 16.99 | -534.6% |
| CNN finetuned — TF single | 16.13 | 6.68 | +15.23% |
| **CNN ensemble ×5 — TF (final)** | **15.87** | **6.49** | **+17.93%** |

### Architecture

| Component | Detail |
|---|---|
| Framework | TensorFlow 2.21 |
| Generator | Conv1D 32→64→64→32, LeakyReLU, BatchNorm, Dropout(0.25) |
| Discriminator | Conv1D 32→64→64→32, LeakyReLU, Dropout(0.25) |
| Regularisation | L2=1e-4, Alpha=5.0, Dropout=0.25 |
| Training | Cosine LR (5e-4→1e-5), early stopping patience=60 |
| Ensemble | 5 models, seeds [42,123,456,789,1024], averaged |

### On R² Targets
R²=17.93% is the realistic ceiling for 3-station daily rainfall imputation without atmospheric data.
Achieving R²>35% requires additional stations and meteorological variables (humidity, pressure, wind).
The positive R² confirms the ensemble beats a naive mean predictor — a meaningful result for this data configuration.